# 第 4 周练习：Python → C++ 代码转换器

## 练习目标（理念）

用前沿大语言模型（LLM）把 Python 代码转成可编译、偏高性能的 C++，并对比多家模型的输出质量与耗时。

这本笔记本会做这些事：

- 用 **GPT-4o**、**Claude Sonnet**、**Gemini** 把一组 Python 函数转成 C++
- **流式（streaming）** 生成，并用 Markdown 语法高亮展示 C++
- 可选：本机用 `g++` 编译并跑一遍，做简易基准/正确性检查
- 打印各模型的生成耗时，方便并排对比

## 和本课第 4 周的关系

| 本课概念 | 本练习里你会看到 |
|----------|------------------|
| 多模型 / 多供应商 | OpenAI + Anthropic + Google，都走 OpenAI 兼容客户端 |
| `system` / `user` messages | `SYSTEM_PROMPT` 定工程师角色；user 放「请转换这段 Python」 |
| 流式输出 `stream=True` | `stream_cpp()` 边收边 `update_display` |
| 把生成结果接到工具链 | `compile_and_run()` 用 `g++ -O2` 验证 |

## 怎么跑

1. 从上到下依次运行每个单元格（Shift+Enter）
2. `.env` 里准备好 `OPENAI_API_KEY`、`ANTHROPIC_API_KEY`、`GOOGLE_API_KEY`（缺哪个就跳过哪个模型）
3. 想本地验证编译：本机需安装 `g++`（Windows 可用 MSYS2 / WSL）

**作者：** Uchebuzz (uche.buzugbe@analyticsintelligence.com)


In [ ]:
# ========== 导入：后面转换、流式展示、本地编译都会用到 ==========

# 标准库 os：读环境变量（Environment Variables），例如各家 API Key
import os
# 标准库 time：用 perf_counter 精确计时，对比各模型生成耗时
import time
# 标准库 subprocess：在本机调用 g++ 编译、再运行生成的可执行文件
import subprocess
# 标准库 tempfile：临时目录写 .cpp，跑完自动清理，不污染工作区
import tempfile
# pathlib.Path：用面向对象方式拼临时文件路径
from pathlib import Path
# load_dotenv：把 .env 里的密钥读进环境变量，避免把密钥写进代码
from dotenv import load_dotenv
# OpenAI 客户端：本练习三家模型都用 OpenAI 兼容接口（换 base_url）
from openai import OpenAI
# IPython 展示工具：Markdown 渲染、display / update_display 做流式刷新
from IPython.display import Markdown, display, update_display


In [ ]:
# ========== 常量：模型名 + 系统提示词集中管理 ==========

# ── 模型名（改这里即可切换版本；字符串必须和供应商支持的 model id 一致）──
# OpenAI 云端多模态旗舰之一：代码生成质量通常很稳
OPENAI_MODEL   = "gpt-4o"
# Anthropic Claude：日期后缀是具体快照版本号，勿随意改译
CLAUDE_MODEL   = "claude-sonnet-4-5-20250929"
# Google Gemini：Flash 系列偏快，适合对比延迟
GEMINI_MODEL   = "gemini-2.0-flash"

# ── 系统提示词（发给模型的指令，必须保持英文，改译会改变生成行为）──
SYSTEM_PROMPT = """\
You are an expert C++ engineer. When given Python source code, produce the equivalent
C++ (C++17) that:
  • compiles cleanly with g++ -std=c++17 -O2
  • preserves the exact semantics of the Python version
  • uses idiomatic modern C++ (auto, range-for, STL algorithms where appropriate)
  • includes a main() that runs the function and prints its result to stdout

Return ONLY the C++ source code — no markdown fences, no explanation.
"""


In [ ]:
# ========== 环境变量 + 三个 OpenAI 兼容客户端 ==========

# override=True：.env 里的值覆盖进程里已有同名环境变量（本练习原逻辑如此）
load_dotenv(override=True)

# 分别读取三家密钥；缺了后面循环会打印 NOT SET，对应客户端调用会失败
openai_key     = os.getenv("OPENAI_API_KEY")
anthropic_key  = os.getenv("ANTHROPIC_API_KEY")
google_key     = os.getenv("GOOGLE_API_KEY")

# 官方 OpenAI：默认 base_url，只需 api_key
openai_client  = OpenAI(api_key=openai_key)
# Anthropic：走 OpenAI 兼容网关，需把 base_url 指到 Anthropic 的 /v1/
claude_client  = OpenAI(
    api_key  = anthropic_key,
    base_url = "https://api.anthropic.com/v1/"
)
# Google Gemini：同样用 OpenAI 兼容层，base_url 指向 Google 的 openai 兼容路径
gemini_client  = OpenAI(
    api_key  = google_key,
    base_url = "https://generativelanguage.googleapis.com/v1beta/openai/"
)

# 启动自检：只打印密钥前 6 个字符，方便确认「读到了」又不把整串密钥打到屏幕
for label, key in [("OpenAI", openai_key), ("Anthropic", anthropic_key), ("Google", google_key)]:
    # 有密钥 → begins xxxxxx；没有 → NOT SET（英文状态串保持原样，便于检索日志）
    status = f"begins {key[:6]}" if key else "NOT SET"
    print(f"{label}: {status}")


## 要转换的 Python 函数

下面放了两个**计算密集型**经典例子：筛素数、朴素矩阵乘法——正好适合拿来对比「Python 语义 → C++ 性能」的转换质量。


In [ ]:
# ========== 示例源码：字符串里是「待转换的 Python」，不是本格要执行的逻辑 ==========

# 三引号字符串原样保留：发给模型的 Python 源码，翻译/改写会改变转换任务本身
# 例 1：埃拉托斯特尼筛法；例 2：朴素矩阵乘法 O(n³)
# 字典 PYTHON_EXAMPLES：展示名 → 源码字符串，主循环按 items() 遍历

PYTHON_SIEVE = """\
def sieve_of_eratosthenes(limit: int) -> list[int]:
    """Return all prime numbers up to `limit` (inclusive)."""
    is_prime = [True] * (limit + 1)
    is_prime[0] = is_prime[1] = False
    for i in range(2, int(limit ** 0.5) + 1):
        if is_prime[i]:
            for j in range(i * i, limit + 1, i):
                is_prime[j] = False
    return [i for i, p in enumerate(is_prime) if p]

primes = sieve_of_eratosthenes(1_000_000)
print(f"Found {len(primes)} primes up to 1,000,000")
print(f"Largest prime: {primes[-1]}")
"""

# ── 例二：矩阵乘法────────────────────────────────────────────────
PYTHON_MATMUL = """\
def matmul(A: list[list[float]], B: list[list[float]]) -> list[list[float]]:
    """Naive O(n^3) matrix multiplication."""
    n = len(A)
    C = [[0.0] * n for _ in range(n)]
    for i in range(n):
        for k in range(n):
            for j in range(n):
                C[i][j] += A[i][k] * B[k][j]
    return C

N = 200
A = [[float(i + j) for j in range(N)] for i in range(N)]
B = [[float(i * j + 1) for j in range(N)] for i in range(N)]
C = matmul(A, B)
print(f"C[0][0] = {C[0][0]:.4f}")
print(f"C[{N-1}][{N-1}] = {C[N-1][N-1]:.4f}")
"""

PYTHON_EXAMPLES = {
    "Sieve of Eratosthenes": PYTHON_SIEVE,
    "Matrix Multiply (200×200)": PYTHON_MATMUL,
}


In [ ]:
# ========== 核心：对流式调用某一模型，边生成边刷新 Markdown 代码块 ==========

def stream_cpp(client: OpenAI, model: str, python_code: str, label: str) -> str:
    """Stream C++ output from the given model and return the full text."""
    # 先显示标题：哪个供应商标签 → 哪个 model id
    display(Markdown(f"### `{label}` → `{model}`"))
    # display_id=True：拿到可更新的句柄，后面用 update_display 原地刷新（打字机效果）
    display_handle = display(Markdown("*Generating…*"), display_id=True)

    # Chat Completions：system 定角色，user 放「Convert this Python…」+ 源码；stream=True 逐块返回
    stream = client.chat.completions.create(
        model=model,
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user",   "content": f"Convert this Python code to C++:\n\n{python_code}"},
        ],
        stream=True,
    )

    # chunks：累积每个 delta 文本片，最后 join 成完整 C++
    chunks = []
    for chunk in stream:
        # delta.content 可能为 None（例如结束片），用 or "" 避免 TypeError
        delta = chunk.choices[0].delta.content or ""
        chunks.append(delta)
        # 把目前已生成的内容包进 ```cpp 围栏，刷新同一 display 区域
        update_display(
            Markdown(f"```cpp\n{''.join(chunks)}\n```"),
            display_id=display_handle.display_id,
        )

    # 返回完整字符串，供后续 compile_and_run 使用
    return "".join(chunks)


In [ ]:
# ========== 可选工具链：把生成的 C++ 写临时文件 → g++ 编译 → 运行 ==========

def compile_and_run(cpp_code: str) -> tuple[bool, str]:
    """
    Write cpp_code to a temp file, compile with g++, run it.
    Returns (success, output_or_error).
    """
    # TemporaryDirectory：with 结束自动删目录，避免留下 main.cpp / 可执行文件
    with tempfile.TemporaryDirectory() as tmpdir:
        # 源文件与可执行文件路径
        src  = Path(tmpdir) / "main.cpp"
        exe  = Path(tmpdir) / "main"
        # 写入模型生成的 C++（utf-8）
        src.write_text(cpp_code, encoding="utf-8")

        # g++：C++17 + -O2 优化；capture_output 把编译错误收进 stderr
        compile_result = subprocess.run(
            ["g++", "-std=c++17", "-O2", "-o", str(exe), str(src)],
            capture_output=True, text=True
        )
        # 非 0 表示编译失败：把 stderr 原样返回给上层展示
        if compile_result.returncode != 0:
            return False, compile_result.stderr

        # 编译成功则运行；timeout=30 防止死循环拖死笔记本
        run_result = subprocess.run(
            [str(exe)], capture_output=True, text=True, timeout=30
        )
        # 成功时返回程序 stdout
        return True, run_result.stdout


## 从每个模型生成 C++

对每个 Python 示例，依次调用 OpenAI / Claude / Gemini，流式展示结果并记录耗时。


In [ ]:
# ========== 主循环：示例 × 模型，收集 results[示例名][标签] = C++ 文本 ==========

# 三元组列表：(客户端, model id, 人类可读标签)
MODELS = [
    (openai_client,  OPENAI_MODEL,  "OpenAI"),
    (claude_client,  CLAUDE_MODEL,  "Claude"),
    (gemini_client,  GEMINI_MODEL,  "Gemini"),
]

# 嵌套字典：外层 key=示例名，内层 key=模型标签，value=生成的 C++ 源码
results: dict[str, dict[str, str]] = {}  # results[example][model_label] = cpp_code

# 先遍历每个 Python 示例
for example_name, python_code in PYTHON_EXAMPLES.items():
    # 展示分隔线 + 示例名 + 原始 Python（方便对照生成结果）
    display(Markdown(f"---\n## {example_name}\n\n**Python source:**\n```python\n{python_code}\n```"))
    # 为该示例准备空的内层字典
    results[example_name] = {}

    # 再遍历三家模型
    for client, model, label in MODELS:
        try:
            # perf_counter：单调时钟，适合测墙钟耗时
            t0  = time.perf_counter()
            # 流式生成完整 C++
            cpp = stream_cpp(client, model, python_code, label)
            elapsed = time.perf_counter() - t0
            # 存进 results，供下一格编译对比
            results[example_name][label] = cpp
            # 打印生成耗时（秒，一位小数）
            display(Markdown(f"*Generated in {elapsed:.1f}s*"))
        except Exception as exc:
            # 某一家失败不中断整本笔记本：把异常展示出来即可
            display(Markdown(f"**Error from {label}:** `{exc}`"))


## 编译并运行（可选）

需要本机已安装 `g++`。Windows 可用 [MSYS2](https://www.msys2.org/) 或 WSL。

理念：不只「看起来像 C++」，还要用编译器验证能否跑通。


In [ ]:
# ========== 对比表：每个示例下，各模型能否编译、stdout 长什么样 ==========

# 遍历前面收集的 results
for example_name, model_outputs in results.items():
    # 小节标题：当前示例名
    display(Markdown(f"### {example_name}"))
    # 先准备 Markdown 表格表头两行
    rows = ["| Model | Compiled? | Output |",
            "|-------|-----------|--------|"]

    # 对每个模型的 C++ 尝试编译运行
    for label, cpp_code in model_outputs.items():
        try:
            ok, output = compile_and_run(cpp_code)
            # 成功/失败用 emoji 标记（展示用，不影响逻辑）
            status = "✅" if ok else "❌"
            # 成功取 stdout 前 120 字；失败取错误信息前 120 字（换行压成空格便于表格）
            snippet = output.strip().replace("\n", " ")[:120] if ok else output.strip()[:120]
        except FileNotFoundError:
            # 系统找不到 g++ 可执行文件时走这里
            status  = "⚠️"
            snippet = "`g++` not found — paste code into [godbolt.org](https://godbolt.org)"
        except Exception as exc:
            # 其它异常（超时、权限等）统一记失败
            status  = "❌"
            snippet = str(exc)[:120]

        # 追加一行表格
        rows.append(f"| {label} | {status} | {snippet} |")

    # 把整张表渲染成 Markdown
    display(Markdown("\n".join(rows)))


## 思考（对照实验后的观察）

三个前沿模型都能从简单 Python 生成可用、偏惯用的 C++。常见观察：

- **GPT-4o**：往往更干净地用 STL 容器，输出更简洁
- **Claude Sonnet**：常加内嵌注释解释算法选择，可读性更好
- **Gemini Flash**：通常响应最快，大多可编译，但偶尔会漏 `#include` 头文件

生产建议：把编译错误再喂回模型，做「自我纠正」循环（self-correction loop），比一次性生成更稳。
